# Discovery

RocketPy imports

In [ ]:
from rocketpy import (
    Environment,
    Rocket,
    Flight,
)
import datetime
from rocketpy.motors import Fluid, CylindricalTank, MassFlowRateBasedTank, LiquidMotor

Defining Environment

In [ ]:
flight_date = datetime.date(2026, 8, 15)
env = Environment(latitude=28.6, longitude=-80.6, elevation=195)

# env.set_date((flight_date.year, flight_date.month, flight_date.day, 12))
# env.set_atmospheric_model(type="Forecast", file="GFS")
env.set_atmospheric_model("custom_atmosphere", wind_u=8.3333, wind_v=0)

Defining the main parameters of the rocket:
- Radius (m)
- Mass without motors (kg)
- Moment of inertia (using formula of a solid cylinder)
- Drag coefficient as a function of Mach number when motor is off/on
- Position of center of mass without motors (m)
- Orientation of rocket

In [ ]:
rocket = Rocket(
    radius=15.2 / 200,
    mass=36.454,  # rocket's mass without the motor in kg
    inertia=( 
        43.919, # (1/4) * mass * radius^2 + (1/12) * mass * length^2
        43.919, # (1/4) * mass * radius^2 + (1/12) * mass * length^2
        0.105, # (1/2) * mass * radius^2
    ),  # in relation to the rocket's center of mass without motor
    power_off_drag="DragCurve.csv",
    power_on_drag="DragCurve.csv",
    center_of_mass_without_motor=1.98,
    coordinate_system_orientation="tail_to_nose",
)

Adding the rocket's components
- Rail buttons
- Nose cone
- Fins
- Parachutes

In [ ]:
""" Adding the rail guide """
rail_buttons = rocket.set_rail_buttons(
    upper_button_position=0.0818,
    lower_button_position=0.6182,
    angular_position=45,
)

""" Adding aerodynamic components """

""" Nose cone """
nose_cone = rocket.add_nose(length=0.451, kind="ogive", position=3.8)

""" Fins """
fin_set = rocket.add_trapezoidal_fins(
    n=3,
    root_chord=0.203,
    tip_chord=0.178,
    span=0.14,
    cant_angle=0,
    sweep_length=0.0509,
    airfoil=("Airfoil.csv", "radians"),
    position= 0.218,)

""" Adding parachute """

main = rocket.add_parachute(
    name="main",
    cd_s=16.074, # reference area * drag coefficient
    trigger=610,  # ejection altitude in meters
    sampling_rate=105,
)

drogue = rocket.add_parachute(
    name="drogue",
    cd_s=0.5905, # reference area * drag coefficient
    trigger="apogee",  # ejection altitude in meters
    sampling_rate=105,
    lag=2,
)

Setting motor parameters

In [ ]:
# OUTDATED! liquid motor example.

from math import exp

""" Defining oxidizer and fuel """
oxidizer_liquid = Fluid(name="Liquid N2O", density=1220)  # kg/m3

oxidizer_gas = Fluid(name="Gas N2O", density=1.9277)  # kg/m3

fuel_liq = Fluid(name="Liquid ethanol", density=789)  # kg/m3

fuel_gas = Fluid(name="Gas ethanol", density=1.59)  # kg/m3

""" Creating the tank """
tanks_geometry = CylindricalTank(
    radius=0.1, height=1.2, spherical_caps=True  # meters  # meters
)


""" Defining flow rate of the oxidizer """

def mass_flow_rate_function(t):  # load .csv table of time and flow rate
    return 32 / 3 * exp(-0.25 * t)


oxidizer_tank = MassFlowRateBasedTank(
    name="oxidizer tank",
    geometry=tanks_geometry,
    flux_time=5,  # seconds
    initial_liquid_mass=32,  # kg
    initial_gas_mass=0.01,  # kg
    liquid_mass_flow_rate_in=0,
    liquid_mass_flow_rate_out=mass_flow_rate_function,
    gas_mass_flow_rate_in=0,
    gas_mass_flow_rate_out=0,
    liquid=oxidizer_liquid,
    gas=oxidizer_gas,
)

""" Defining the flow rate of the fuel """


def liquid_mass_flow_rate_function(t):  # load .csv table of time and flow rate
    return 21 / 3 * exp(-0.25 * t)


def gas_mass_flow_rate_function(t):  # load .csv table of time and flow rate
    return 0.01 / 3 * exp(-0.25 * t)


fuel_tank = MassFlowRateBasedTank(
    name="fuel tank",
    geometry=tanks_geometry,
    flux_time=5,  # seconds
    initial_liquid_mass=21,  # kg
    initial_gas_mass=0.01,  # kg
    liquid_mass_flow_rate_in=0,
    liquid_mass_flow_rate_out=liquid_mass_flow_rate_function,
    gas_mass_flow_rate_in=0,
    gas_mass_flow_rate_out=gas_mass_flow_rate_function,
    liquid=fuel_liq,
    gas=fuel_gas,
)

""" Defining the thrust curve"""


def thrust_curve_function(t):  # load .csv table of time and thrust
    return 10000 - 100 * t**2


""" Assembling liquid motor """
liquid_motor = LiquidMotor(
    thrust_source=thrust_curve_function,
    dry_mass=2,  # kg
    dry_inertia=(0.125, 0.125, 0.002),  # kg/m2
    nozzle_radius=0.075,
    center_of_dry_mass_position=1.75,
    nozzle_position=0,
    burn_time=5,  # seconds
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

""" Add tanks to the motor """
liquid_motor.add_tank(
    tank=oxidizer_tank,
    position=1.0,  # meters
)

liquid_motor.add_tank(
    tank=fuel_tank,
    position=2.5,  # meters
)

# liquid_motor.info()
# liquid_motor.draw()


Defining Flight

In [ ]:
flight = Flight(
    rocket=rocket,
    environment=env,
    rail_length=9.21,
    inclination=85.0,
    heading=270.0,
)